# Mixing runtime

[The comparison notebook](03-MixingComparison.ipynb) times the matrix alone. This one times a full solve. It is not a parity check, and it does not change the golden path: `run()` still calls `get_norm_c_matrix`.

Use the **vacf (pixi)** kernel, as in [the parity notebook](01-InspectRun.ipynb).

The bar is the mean time of a solve after compilation, across six parameter draws. Each case is solved once and that call is thrown away: it pays for JIT. The six draws are then solved again, and those six times are the mean, the standard deviation on the bar, and the `with JIT` column. That column spreads the discarded call across the six. It is not the bar.

Each draw replaces every key in `PARAM_BOUNDS` with a uniform value on that interval, seed 0. The other keys stay at `BASE_PARAMS`. Every summer4 case uses those same six draws. Summer3 draws with the same seed from the snapshot, whose bounds match.

- **summer3 golden** is the snapshot model, built the way `reference/make_golden.py` builds it, without writing parquets. Dopri5, initial step `0.1`, `rtol=atol=1e-5`, `dtmax=1`, `max_steps=4000`. The notebook builds a `Diffrax` backend with that `PIDController` (summer4 `run` has no `dtmax=` keyword).
- **summer4 golden-matching** is `run(params, match_golden=True)`. The three summer3 bugs stay in.
- **summer4 corrected, baseline** is `run(params)` with the same mixing function.
- **stepwise fixed** folds `bg_mixing`, `a_spread` and `pc_strength` into a year table for that draw, then the solve only indexes. The table is an input, so a new draw does not compile again. The fold is inside the timed call.
- **stepwise free** reads the three scalars from the parameters on every stage and rebuilds one year plus the spectral radius.
- **smooth fixed** and **smooth free** are the same split for the PCHIP matrix. Fixed folds the scalars into the stored curves, then each stage evaluates those curves and the eigenproblem. Free rebuilds the assortative kernel from `a_spread` as well.
- **spline fixed** and **spline free** are the same split again for `prepare_spline`, which normalises at each calendar year and runs a cubic through the normalised matrices. Fixed hands the solve one fitted curve; the stage is a gather and a Horner cubic with no eigenproblem in it. Free refits the four knots the segment reads, inside every stage.

Read this table together with the two sections after it. A plain `compiled.run` call spends about a second before it integrates anything, on work that has nothing to do with the mixing matrix, and that constant is large enough to hide the differences this notebook is about.

Both sides use that same controller. Summer4 is handed the numbers explicitly; summer3 inherits them. The summer4 cases share one process, so a later compilation can reuse kernels from an earlier one. Each case still pays for its own warmup before its six timed solves.

Stepwise is built for 1800–2101, the span the solver actually queries. Before the weight table starts, weights and populations clip and fertility still moves, which is what `get_norm_c_matrix` does. Smooth is fit on the weight years and is constant before 1950, so it is a different vector field and its step count is not the baseline's. Building those demographic records is once per case. That time is printed and is not in the mean.


## Mean time per solve, after JIT

The summer3 number comes from the reference environment's Python, running `reference/time_solve.py`. This kernel cannot import summer3. Model construction is outside that timer.

Every summer4 case is compiled before any of them is timed, and the six draws are then timed round-robin across the cases. Building them in order and timing each as it is built does not work here: this kernel ends up holding eight compiled models, and the cost of that falls on whichever case happens to be built last rather than on the one that causes it.

In [1]:
import os
import subprocess
import time
from pathlib import Path

import jax
import numpy as np
import plotly.graph_objects as go

from vacf.constants import END_TIME, START_TIME
from vacf.contact import prepare_smooth, prepare_spline, prepare_stepwise
from vacf.contact.pchip import eval_jax
from vacf.contact.stepwise import year_index_jax
from vacf.mixing import normalise_by_spectral_radius
from vacf.model import STATIC, compile_model, run, solve
from vacf.parameters import BASE_PARAMS, PARAM_BOUNDS

N_SETS = 6
SEED = 0

# Same controller as the tb_macroeconomics notebooks: max_steps=4000, and the
# summer3 runner defaults Dopri5, dt0=0.1, rtol=atol=1e-5, dtmax=1.
from vacf.solvers import dopri5

SOLVER = dict(
    dt=0.1,
    solver=dopri5(rtol=1e-5, atol=1e-5, dtmax=1.0),
    max_steps=4_000,
)

ROOT = Path.cwd().resolve()
if not (ROOT / "pixi.toml").is_file():
    ROOT = ROOT.parent


def draw_params(n_sets, seed):
    rng = np.random.default_rng(seed)
    drawn = []
    for _ in range(n_sets):
        params = dict(BASE_PARAMS)
        for name, (low, high) in PARAM_BOUNDS.items():
            params[name] = float(rng.uniform(low, high))
        drawn.append(params)
    return drawn


def seconds(fn):
    started = time.perf_counter()
    jax.block_until_ready(fn())
    return time.perf_counter() - started


def summarise(label, jit_s, post):
    post = np.asarray(post, dtype=np.float64)
    mean = float(post.mean())
    std = float(post.std(ddof=1))
    with_jit = (jit_s + float(post.sum())) / post.size
    rows.append((label, jit_s, mean, std, with_jit))


# Each summer4 case is registered, not timed, until every one of them has been
# compiled. Timing them in the order they are built makes the last case look
# slow: this kernel holds eight compiled models at once, and the cost of that
# lands on whoever runs last, not on whoever deserves it.
cases = []


def register(label, once):
    cases.append((label, once))


def time_registered():
    """One discarded solve each for the compile, then round-robin the draws."""
    compile_s = {}
    for label, once in cases:
        compile_s[label] = seconds(lambda once=once: once(sets[0]))
    post = {label: [] for label, _ in cases}
    for params in sets:
        for label, once in cases:
            post[label].append(seconds(lambda once=once, params=params: once(params)))
    for label, _ in cases:
        summarise(label, compile_s[label], post[label])


sets = draw_params(N_SETS, SEED)
rows = []

reference_python = ROOT / ".pixi" / "envs" / "reference" / "bin" / "python"
env = os.environ.copy()
for key in (
    "PYTHONPATH",
    "PYTHONHOME",
    "VIRTUAL_ENV",
    "CONDA_PREFIX",
    "CONDA_DEFAULT_ENV",
    "MPLBACKEND",
):
    env.pop(key, None)
env["MPLBACKEND"] = "Agg"
proc = subprocess.run(
    [str(reference_python), str(ROOT / "reference" / "time_solve.py")],
    cwd=ROOT,
    capture_output=True,
    text=True,
    env=env,
)
if proc.returncode != 0:
    raise RuntimeError(proc.stderr or proc.stdout)
jit_line = [item for item in proc.stdout.splitlines() if item.startswith("JIT ")][-1]
post_line = [item for item in proc.stdout.splitlines() if item.startswith("POST ")][-1]
summarise(
    "summer3 golden",
    float(jit_line.split()[1]),
    [float(item) for item in post_line.split()[1:]],
)


register(
    "summer4 golden-matching",
    lambda params: run(params, match_golden=True, **SOLVER)["compartments"].values.data,
)
register(
    "summer4 corrected, baseline",
    lambda params: run(params, **SOLVER)["compartments"].values.data,
)

arrays = (
    STATIC.age_weights,
    STATIC.weight_ends,
    STATIC.pops,
    STATIC.pop_ends,
    STATIC.fert,
    STATIC.fert_ends,
)
span = np.arange(START_TIME, END_TIME + 1.0)


def mixing_free(contact):
    def mixing(calendar, params):
        return contact.matrix(
            calendar, params["bg_mixing"], params["a_spread"], params["pc_strength"]
        )

    return mixing


def mixing_table(contact):
    def mixing(calendar, params):
        index = year_index_jax(calendar, contact.year0, contact.year1)
        return params["contact_table"][index]

    return mixing


def mixing_spline(contact):
    def mixing(calendar, params):
        return params["contact_curve"].at(calendar)

    return mixing


def mixing_curves(contact):
    def mixing(calendar, params):
        mixed = eval_jax(calendar, contact.knot_x, params["offset_y"], params["offset_m"])
        pop = eval_jax(calendar, contact.knot_x, contact.pop_y, contact.pop_m)
        return normalise_by_spectral_radius(mixed * pop[None, :])

    return mixing


def with_table(contact, params):
    ready = contact.materialize(params["bg_mixing"], params["a_spread"], params["pc_strength"])
    out = dict(params)
    out["contact_table"] = ready.table
    return out


def with_curve(contact, params):
    ready = contact.materialize(params["bg_mixing"], params["a_spread"], params["pc_strength"])
    out = dict(params)
    out["contact_curve"] = ready.curve
    return out


def with_curves(contact, params):
    ready = contact.materialize(params["bg_mixing"], params["a_spread"], params["pc_strength"])
    out = dict(params)
    out["offset_y"] = ready.offset_y
    out["offset_m"] = ready.offset_m
    return out


def run_variant(label, mixing, attach):
    compiled = compile_model(mixing=mixing)
    register(label, lambda params: solve(compiled, attach(params), **SOLVER)["compartments"].values.data)


started = time.perf_counter()
step = prepare_stepwise(*arrays, years=span)
print(f"prepare stepwise: {time.perf_counter() - started:.3f}s")
started = time.perf_counter()
smooth = prepare_smooth(*arrays)
print(f"prepare smooth: {time.perf_counter() - started:.3f}s")
started = time.perf_counter()
spline = prepare_spline(*arrays, years=span)
print(f"prepare spline: {time.perf_counter() - started:.3f}s")
run_variant(
    "summer4 corrected, stepwise fixed",
    mixing_table(step),
    lambda params: with_table(step, params),
)
run_variant(
    "summer4 corrected, stepwise free",
    mixing_free(step),
    lambda params: params,
)
run_variant(
    "summer4 corrected, smooth fixed",
    mixing_curves(smooth),
    lambda params: with_curves(smooth, params),
)
run_variant(
    "summer4 corrected, smooth free",
    mixing_free(smooth),
    lambda params: params,
)
run_variant(
    "summer4 corrected, spline fixed",
    mixing_spline(spline),
    lambda params: with_curve(spline, params),
)
run_variant(
    "summer4 corrected, spline free",
    mixing_free(spline),
    lambda params: params,
)

time_registered()

print(f"{'case':<40} {'jit':>8} {'mean':>8} {'std':>8} {'with JIT':>8}")
for label, jit_s, mean, std, with_jit in rows:
    print(f"{label:<40} {jit_s:8.3f} {mean:8.3f} {std:8.3f} {with_jit:8.3f}")

figure = go.Figure(
    go.Bar(
        y=[label for label, *_ in rows],
        x=[mean for _, _, mean, _, _ in rows],
        orientation="h",
        error_x=dict(
            type="data",
            array=[std for _, _, _, std, _ in rows],
            visible=True,
        ),
    )
)
figure.update_layout(
    xaxis_title="Mean post-JIT solve (seconds)",
    margin=dict(l=280, r=20, t=20, b=40),
    height=420,
)
figure.show()


prepare stepwise: 0.114s


prepare smooth: 0.269s
prepare spline: 0.072s


case                                          jit     mean      std with JIT
summer3 golden                             10.777    2.457    0.170    4.253
summer4 golden-matching                     2.882    1.579    0.628    2.060
summer4 corrected, baseline                 2.200    1.590    0.539    1.957
summer4 corrected, stepwise fixed           1.845    1.205    0.567    1.512
summer4 corrected, stepwise free            1.875    1.502    0.673    1.815
summer4 corrected, smooth fixed             1.879    1.384    0.625    1.697
summer4 corrected, smooth free              2.741    2.026    0.922    2.483
summer4 corrected, spline fixed             2.184    1.678    1.056    2.042
summer4 corrected, spline free              2.078    1.464    0.599    1.811


## What a repeated solve costs

Every number here is regenerated when the notebook runs; the ordering is what to read, not the third digit.

Folding the three scalars once per draw is what moves the solve. Leaving them free means a stage rebuilds part of the matrix, and for the smooth and spline variants that is most of the advantage given back.

The three fixed rows — stepwise, smooth and spline — land close together, and that is not because the matrices cost the same. About a second of each of these calls is spent before the solver starts, on work that is identical in every row. The next section takes that out, and the three then separate by more than a factor of two.

Building the demographic records is once per process and is not in the mean; the per-draw fold is. Spline prepares fastest of the three because it needs one record per calendar year, where smooth samples four times a year.

## Most of `compiled.run` is not the solve

The means above are not all integration. About 1.6 seconds of every `compiled.run` call is equinox re-tracing `diffeqsolve`: `summer4.solvers.diffrax_backend` rebuilds the `SubSaveAt` closures on each call, `equinox.filter_jit` treats a function as static and keys it by identity, so the cache misses and the whole solve is traced again. Nothing about the model or the mixing matrix changes that, and it is the same constant in every row, so it compresses all of them towards each other.

A calibration does not pay it: NUTS traces its log-density once and calls the compiled thing thereafter. The cell below wraps the identical solve in one outer `jax.jit` so it is traced once, which is the cost that actually matters here.

The profile that identified it is `cProfile` over three repeat calls, with `jax._src.pjit._trace_for_jit` holding about a second per call. Worth reporting upstream; it is a harness issue, not a model one.

In [2]:
import jax.numpy as jnp

from vacf.constants import N_TIMES
from vacf.model import save_plan

from vacf.solvers import dopri5

SOLVER = dict(
    t0=0.0,
    t1=float(N_TIMES),
    dt=0.1,
    solver=dopri5(rtol=1e-5, atol=1e-5, dtmax=1.0),
    max_steps=4_000,
)


def theta(params):
    return jnp.asarray(
        [params["bg_mixing"], params["a_spread"], params["pc_strength"]], dtype=jnp.float64
    )


def raw_and_jitted(label, mixing, attach):
    """Time the same solve through `compiled.run` and inside one outer jit."""
    compiled = compile_model(mixing=mixing)
    kwargs = dict(SOLVER, save=save_plan(compiled))

    @jax.jit
    def wrapped(scalars):
        params = dict(BASE_PARAMS)
        params["bg_mixing"], params["a_spread"], params["pc_strength"] = scalars
        # The fold is inside the traced region, so the draw pays for it here
        # and the solve sees a table or a curve, not the three scalars.
        return compiled.run(attach(params), **kwargs)["compartments"].values.data

    def raw(params):
        return compiled.run(attach(params), **kwargs)["compartments"].values.data

    seconds(lambda: raw(sets[0]))
    raw_times = [seconds(lambda params=params: raw(params)) for params in sets]
    seconds(lambda: wrapped(theta(sets[0])))
    jit_times = [seconds(lambda params=params: wrapped(theta(params))) for params in sets]
    result = compiled.run(attach(sets[0]), **kwargs)
    return (
        label,
        float(np.median(raw_times)),
        float(np.median(jit_times)),
        int(result.solver.num_steps),
        int(result.solver.num_rejected_steps),
    )


split = [
    raw_and_jitted("baseline", None, lambda params: params),
    raw_and_jitted(
        "stepwise fixed", mixing_table(step), lambda params: with_table(step, params)
    ),
    raw_and_jitted(
        "smooth fixed", mixing_curves(smooth), lambda params: with_curves(smooth, params)
    ),
    raw_and_jitted(
        "spline fixed", mixing_spline(spline), lambda params: with_curve(spline, params)
    ),
]

print(f"{'case':<18}{'run()':>9}{'outer jit':>11}{'steps':>8}{'rejected':>10}")
for label, raw_s, jit_s, steps, rejected in split:
    print(f"{label:<18}{raw_s:9.3f}{jit_s:11.3f}{steps:8d}{rejected:10d}")
constant = split[0][1] - split[0][2]
print(f"\nthe re-trace is worth about {constant:.2f} s per call, and it is in every row")

figure = go.Figure()
figure.add_trace(
    go.Bar(name="solve", x=[label for label, *_ in split], y=[jit_s for _, _, jit_s, _, _ in split])
)
figure.add_trace(
    go.Bar(
        name="re-trace",
        x=[label for label, *_ in split],
        y=[raw_s - jit_s for _, raw_s, jit_s, _, _ in split],
    )
)
figure.update_layout(
    barmode="stack",
    yaxis_title="Seconds per call",
    title="What a compiled.run call spends its time on",
    height=400,
)
figure.show()


case                  run()  outer jit   steps  rejected
baseline              1.647      0.338    1488       425
stepwise fixed        1.303      0.078    1488       425
smooth fixed          1.352      0.099    1114       266
spline fixed          1.322      0.084    1120       268

the re-trace is worth about 1.31 s per call, and it is in every row


## The rejected-step budget

Under the outer jit the mixing variants separate, and the reason is visible in the last two columns above: how many steps Dopri5 proposes and how many it throws away.

`get_year_index` freezes the matrix on each calendar year, so the baseline field jumps every 1 January. A PID controller assumes a smooth field; each jump is a step proposed, evaluated over six stages, and rejected. At these tolerances the baseline rejects 425 of 1 488 steps, about 30%, and `prepare_stepwise` does not help, because reproducing the baseline exactly means reproducing the jump exactly.

There is a second jump on the same dates. `demography.birth_rate_at` is a `searchsorted` step lookup on the entry rate. Smoothing only that rate barely moves the count (397 rejected). A C1 matrix cuts it to 268, and both together to 197. The matrix jump is the larger of the two; at these tolerances neither fix empties the column.

`demography.smooth_birth_rate_at` is a linear interpolation of the same rates. Nothing calls it. It is a different vector field from the one the goldens record, so this is a measurement, not a proposal.

In [3]:
import vacf.demography as demography
import vacf.model as model_module


def steps_for(mixing, attach, smooth_entry):
    """Step and rejection counts, optionally with a C1 entry rate."""
    original = model_module.birth_rate_at
    if smooth_entry:
        model_module.birth_rate_at = demography.smooth_birth_rate_at
    try:
        compiled = compile_model(mixing=mixing)
        kwargs = dict(SOLVER, save=save_plan(compiled))
        result = compiled.run(attach(sets[0]), **kwargs)
        jax.block_until_ready(result["compartments"].values.data)
        return int(result.solver.num_steps), int(result.solver.num_rejected_steps)
    finally:
        model_module.birth_rate_at = original


fields = [
    ("baseline", None, lambda params: params),
    ("stepwise fixed", mixing_table(step), lambda params: with_table(step, params)),
    ("spline fixed", mixing_spline(spline), lambda params: with_curve(spline, params)),
]

print(f"{'mixing':<16}{'entry rate':<12}{'steps':>8}{'rejected':>10}{'rejected %':>12}")
budget = []
for label, mixing, attach in fields:
    for smooth_entry in (False, True):
        steps, rejected = steps_for(mixing, attach, smooth_entry)
        name = "C1" if smooth_entry else "step"
        print(f"{label:<16}{name:<12}{steps:8d}{rejected:10d}{rejected / steps * 100:11.1f}%")
        budget.append((f"{label}, {name} entry", steps - rejected, rejected))

figure = go.Figure()
figure.add_trace(
    go.Bar(name="accepted", y=[name for name, *_ in budget], x=[a for _, a, _ in budget], orientation="h")
)
figure.add_trace(
    go.Bar(name="rejected", y=[name for name, *_ in budget], x=[r for *_, r in budget], orientation="h")
)
figure.update_layout(
    barmode="stack",
    xaxis_title="Dopri5 steps",
    title="Both 1 January discontinuities have to go, or neither pays",
    margin=dict(l=220, r=20, t=50, b=40),
    height=420,
)
figure.show()


mixing          entry rate     steps  rejected  rejected %


baseline        step            1488       425       28.6%


baseline        C1              1441       397       27.6%


stepwise fixed  step            1488       425       28.6%


stepwise fixed  C1              1441       397       27.6%


spline fixed    step            1120       268       23.9%


spline fixed    C1              1001       197       19.7%


## How loose the controller can be

The table above is the notebook controller: `rtol = atol = 1e-5` and `dtmax = 1`. This section asks how much of the remaining time is the controller being careful. The cap comes off, and both tolerances move together through `1e-5`, `1e-4`, `1e-3`, `1e-2` and `1e-1`. `max_steps` stays 4 000. The initial step stays 0.1.

The clock is the outer jit from the previous section, on `BASE_PARAMS`, not the six draws. Each mixing is compared with itself at the notebook controller, so a loose spline is not being judged against the baseline field. The error is the largest change in any compartment, divided by the larger of the reference value and one person, so an empty compartment cannot dominate. The population column is the same ratio for the sum of compartments at the worst saved year.

A row is marked plausible when that compartment error stays under 1%, the population error stays under 1%, every value is finite, and no compartment drops below minus one person. Those cuts are a reading of this table, not a calibration tolerance.

That flag stays on only for the notebook controller and for the same tolerances with the cap removed. Taking `dtmax` off at `1e-5` does not buy time: the baseline stays at 0.34 s, stepwise at 0.08 s, spline at 0.07 s. The compartment move from dropping the cap is 0.4% for the two discontinuous matrices and 0.25% for the spline.

The next two settings miss the flag by a little and are still the same trajectory. At `1e-4` and `1e-3` the baseline and the stepwise matrix stay within 1.3% of their notebook solve, the population within 0.2%, and no compartment goes negative. The clock bottoms out there, on stepwise at `1e-3`: 0.055 s, against 0.081 s with the notebook controller. The spline is in that group at `1e-4` (0.063 s, just over 1%) and leaves it at `1e-3` (3.9%, and a compartment slightly negative).

`1e-2` is where the solve stops being the notebook solve. The largest compartment moves by 36% and a compartment goes negative. `1e-1` is worse, 175% with compartments hundreds to thousands of people below zero, and slower, because rejected steps climb back up. Opening the controller past `1e-3` does not make this model faster.

The fastest row that still looks like the notebook solve is the stepwise matrix, no cap, `rtol = atol = 1e-3`, at 0.055 s. The strict 1% reading only allows the `1e-5` rows, and those are no faster than the table above. The baseline never gets under about 0.24 s, even on the rows that have already left the trajectory.


In [4]:
from vacf.solvers import dopri5


GRID = (
    ("notebook", 1e-5, 1.0),
    ("1e-5, no cap", 1e-5, None),
    ("1e-4, no cap", 1e-4, None),
    ("1e-3, no cap", 1e-3, None),
    ("1e-2, no cap", 1e-2, None),
    ("1e-1, no cap", 1e-1, None),
)


def _solver(compiled, rtol, dtmax):
    return dict(
        t0=0.0,
        t1=float(N_TIMES),
        dt=0.1,
        solver=dopri5(rtol=float(rtol), atol=float(rtol), dtmax=dtmax),
        max_steps=4_000,
        save=save_plan(compiled),
    )


def _departure(got, ref):
    got = np.asarray(got)
    ref = np.asarray(ref)
    scale = np.maximum(np.abs(ref), 1.0)
    point = np.abs(got - ref) / scale
    pop_ref = ref.sum(axis=1)
    pop_got = got.sum(axis=1)
    pop = np.abs(pop_got - pop_ref) / np.maximum(np.abs(pop_ref), 1.0)
    return float(point.max()), float(pop.max()), bool(np.isfinite(got).all()), float(got.min())


def _loose(label, mixing, attach):
    compiled = compile_model(mixing=mixing)
    scalars = theta(BASE_PARAMS)
    reference = None
    rows = []
    for name, rtol, dtmax in GRID:
        kwargs = _solver(compiled, rtol, dtmax)

        def run_jit(values, kwargs=kwargs):
            params = dict(BASE_PARAMS)
            params["bg_mixing"], params["a_spread"], params["pc_strength"] = values
            data = compiled.run(attach(params), **kwargs)["compartments"].values.data
            return data

        wrapped = jax.jit(run_jit)
        jax.block_until_ready(wrapped(scalars))
        started = time.perf_counter()
        got = wrapped(scalars)
        jax.block_until_ready(got)
        elapsed = time.perf_counter() - started
        if reference is None:
            reference = np.asarray(got)
            max_rel, pop_rel, finite, lowest = 0.0, 0.0, True, float(reference.min())
        else:
            max_rel, pop_rel, finite, lowest = _departure(got, reference)
        result = compiled.run(attach(dict(BASE_PARAMS)), **kwargs)
        steps = int(result.solver.num_steps)
        rejected = int(result.solver.num_rejected_steps)
        plausible = finite and lowest >= -1.0 and max_rel <= 0.01 and pop_rel <= 0.01
        rows.append(
            (label, name, elapsed, steps, rejected, max_rel, pop_rel, lowest, plausible)
        )
    return rows


loose = []
loose += _loose("baseline", None, lambda params: params)
loose += _loose(
    "stepwise fixed", mixing_table(step), lambda params: with_table(step, params)
)
loose += _loose(
    "spline fixed", mixing_spline(spline), lambda params: with_curve(spline, params)
)

print(
    f"{'mixing':<16}{'controller':<14}{'jit s':>8}{'steps':>8}{'rej':>6}"
    f"{'max rel':>10}{'pop rel':>10}{'min':>9}{'ok':>4}"
)
for label, name, elapsed, steps, rejected, max_rel, pop_rel, lowest, plausible in loose:
    flag = "yes" if plausible else "no"
    print(
        f"{label:<16}{name:<14}{elapsed:8.3f}{steps:8d}{rejected:6d}"
        f"{max_rel:10.2e}{pop_rel:10.2e}{lowest:9.2f}{flag:>4}"
    )

order = [name for name, *_ in GRID]
figure = go.Figure()
for label in ("baseline", "stepwise fixed", "spline fixed"):
    subset = [row for row in loose if row[0] == label]
    figure.add_trace(
        go.Bar(name=label, x=[row[1] for row in subset], y=[row[2] for row in subset])
    )
figure.update_layout(
    barmode="group",
    yaxis_title="Outer-jit seconds",
    title="Opening the controller",
    height=420,
)
figure.show()

error = go.Figure()
for label in ("baseline", "stepwise fixed", "spline fixed"):
    subset = [row for row in loose if row[0] == label]
    error.add_trace(
        go.Scatter(
            name=label,
            x=[row[1] for row in subset],
            y=[max(row[5], 1e-16) for row in subset],
            mode="lines+markers",
        )
    )
error.update_layout(
    yaxis_type="log",
    yaxis_title="Max compartment error vs notebook controller",
    title="The reference row is zero; the floor on the axis is 1e-16",
    height=420,
)
error.show()


mixing          controller       jit s   steps   rej   max rel   pop rel      min  ok
baseline        notebook         0.309    1284   365  0.00e+00  0.00e+00    -0.00 yes
baseline        1e-5, no cap     0.307    1245   347  4.08e-03  3.44e-04    -0.00 yes
baseline        1e-4, no cap     0.245     980   257  1.10e-02  1.05e-03    -0.00  no
baseline        1e-3, no cap     0.193     875   201  1.25e-02  1.62e-03    -0.00  no
baseline        1e-2, no cap     0.208     846   183  3.60e-01  5.46e-04    -0.03  no
baseline        1e-1, no cap     0.234    1063   402  1.75e+00  8.65e-04  -911.07  no
stepwise fixed  notebook         0.075    1284   365  0.00e+00  0.00e+00    -0.00 yes
stepwise fixed  1e-5, no cap     0.073    1245   347  4.08e-03  3.44e-04    -0.00 yes
stepwise fixed  1e-4, no cap     0.061     980   257  1.10e-02  1.05e-03    -0.00  no
stepwise fixed  1e-3, no cap     0.052     875   201  1.25e-02  1.62e-03    -0.00  no
stepwise fixed  1e-2, no cap     0.053     846   183  